# Support Ticket Triage: Quick EDA

Load the Kaggle CSV, inspect its schema and label balance, then check text length and representative examples. This export labels tickets with five `Ticket Type` values, not the requested four support queues; the explicit mapping below is a modeling assumption, not original ground truth.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/raw/customer_support_tickets.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data/raw/customer_support_tickets.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}. Download customer_support_tickets.csv from the Kaggle dataset and place it in data/raw/.')

tickets = pd.read_csv(DATA_PATH)
print(f'Rows: {len(tickets):,} | Columns: {len(tickets.columns)}')
display(tickets.head(3))

In [ ]:
print('Columns:', tickets.columns.tolist())
display(tickets.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Ticket Type counts:')
display(tickets['Ticket Type'].value_counts(dropna=False).to_frame('tickets'))
print('Provided Ticket Priority counts (EDA only; urgency is computed from sentiment):')
display(tickets['Ticket Priority'].value_counts(dropna=False).to_frame('tickets'))

tickets['text'] = tickets[['Ticket Subject', 'Ticket Description']].fillna('').agg(' '.join, axis=1).str.strip()
tickets['text_chars'] = tickets['text'].str.len()
display(tickets['text_chars'].describe().to_frame('characters'))

In [ ]:
TYPE_TO_CATEGORY = {
    'billing inquiry': 'Billing',
    'refund request': 'Billing',
    'technical issue': 'Technical Support',
    'cancellation request': 'Account',
    'product inquiry': 'General',
}
tickets['category'] = tickets['Ticket Type'].str.strip().str.lower().map(TYPE_TO_CATEGORY)
if tickets['category'].isna().any():
    raise ValueError('Unmapped Ticket Type values: ' + str(tickets.loc[tickets['category'].isna(), 'Ticket Type'].unique()))
display(tickets['category'].value_counts().to_frame('tickets'))
print('Mapping caveat: categories are inferred from ticket type; Account and General are not native labels in this CSV.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tickets['Ticket Type'].value_counts().plot.bar(ax=axes[0], color='#2f6f73')
axes[0].set_title('Original Ticket Type')
axes[0].set_ylabel('Ticket count')
axes[0].tick_params(axis='x', rotation=35)
tickets['category'].value_counts().reindex(['Billing', 'Technical Support', 'Account', 'General']).plot.bar(ax=axes[1], color='#c36b45')
axes[1].set_title('Mapped four-category target')
axes[1].set_ylabel('Ticket count')
axes[1].tick_params(axis='x', rotation=25)
fig.tight_layout()
plt.show()

display(tickets[['Ticket Type', 'category', 'Ticket Subject', 'Ticket Description']].sample(5, random_state=42))